# CouchDB: load, verify and query the Grab Mobility & Fulfillment database

This notebook runs the CouchDB implementation end to end (report sections 2.3, 3.1 and 3.2):

1. connect to CouchDB and show the environment,
2. *(optional)* load the canonical dataset into CouchDB,
3. verify that CouchDB holds exactly the canonical rows,
4. run each of the 25 essential operations once and check its result,
5. query the database freely (documents, views, Mango),
6. *(optional)* run the benchmark (1 warm-up + 3 measured runs per operation),
7. run the integrity-rule tests and write an audit manifest.

**Audit trail.** Every run writes its evidence to `benchmarks/results/couchdb_<dataset>/`:
the environment and code version (`environment.json`), the load and verification reports,
the result of every operation, the benchmark runs, the test output, and a `MANIFEST.json`
with the SHA-256 of each file. Save the notebook afterwards so its outputs are kept as well.

**Before running**

```bash
cp .env.example .env            # then set the passwords
docker compose --env-file .env -f docker/compose.yaml up -d --build
uv sync --frozen
```

Open this notebook with the project's `.venv` kernel and run the cells from top to bottom.

## Parameters

* `DATA_DIR`: the dataset to load and check against. `data/canonical` is the full dataset
  (2,000,000 orders). For a quick trial on real CouchDB, any folder with the same 13 CSV
  files works, e.g. a sample of every 50th order.
* `RUN_LOAD = True` **drops** the four `grab_*` databases and loads `DATA_DIR` again. For the full
  dataset this takes a while (loading plus building the view indexes).
* `RUN_BENCHMARK = True` runs the benchmark runner over the 25 operations.
* `RUN_TESTS = True` runs the integrity-rule and unit tests (temporary databases).
* `PUBLISH = True` copies the evidence (not the large reference file) to
  `benchmarks/published/couchdb_<dataset>/`, which is tracked by Git.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data' / 'canonical'
RUN_LOAD = False
RUN_BENCHMARK = False
RUN_TESTS = True
PUBLISH = False
PREVIEW_ROWS = 10

DATASET = DATA_DIR.name
OUT_DIR = ROOT / 'benchmarks' / 'results' / f'couchdb_{DATASET}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_FILE = ROOT / 'data' / 'reference' / f'couchdb_reference_{DATASET}.json'
print(f'dataset: {DATA_DIR}\nevidence: {OUT_DIR}')

# Keep the Mac awake while this kernel runs (long loads and benchmarks).
import shutil, subprocess
if shutil.which('caffeinate'):
    subprocess.Popen(['caffeinate', '-i', '-w', str(os.getpid())])

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

## 1. Connect and record the environment

In [ ]:
from couchdb import design, load, reference, setup_db, verify_load
from couchdb.client import CouchClient
from couchdb.queries import OPERATIONS, GrabCouch
from couchdb.settings import DATABASES, get_settings

settings = get_settings()
admin = CouchClient.admin(settings, timeout=None)
app = CouchClient.app(settings, timeout=None)
grab = GrabCouch(app, settings)
DB = {name: settings.db(name) for name in DATABASES}

server = setup_db.wait_until_up(admin)
print(f"CouchDB {server.get('version')} at {settings.url}")
print(f"JavaScript engine: {setup_db.javascript_engine(admin)}")
print(f"Operations run as: {app.user} (admin: {admin.user})")

In [ ]:
import hashlib
import importlib.metadata as metadata
import platform
from datetime import UTC, datetime

def sh(*command):
    """Output of a command, or a note when it is not available on this machine."""
    try:
        return subprocess.run(command, capture_output=True, text=True, timeout=30,
                              cwd=ROOT).stdout.strip()
    except Exception as error:
        return f'unavailable ({type(error).__name__})'

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest_file = DATA_DIR / 'manifest.json'
environment = {
    'run_started_utc': datetime.now(UTC).isoformat(),
    'git_commit': sh('git', 'rev-parse', 'HEAD'),
    'git_branch': sh('git', 'rev-parse', '--abbrev-ref', 'HEAD'),
    'git_uncommitted_changes': sh('git', 'status', '--porcelain', '--untracked-files=no').splitlines(),
    'os': platform.platform(),
    'cpu': sh('sysctl', '-n', 'machdep.cpu.brand_string') if sys.platform == 'darwin' else platform.processor(),
    'cpu_cores': os.cpu_count(),
    'ram_gb': (round(int(sh('sysctl', '-n', 'hw.memsize')) / 2**30, 1)
               if sh('sysctl', '-n', 'hw.memsize').isdigit() else None),
    'python': platform.python_version(),
    'packages': {p: metadata.version(p) for p in ('requests', 'pandas', 'pytest')},
    'docker': sh('docker', 'version', '--format', '{{.Server.Version}}'),
    'docker_resources': sh('docker', 'info', '--format', 'cpus={{.NCPU}} memory_bytes={{.MemTotal}}'),
    'couchdb_version': server.get('version'),
    'couchdb_features': server.get('features'),
    'javascript_engine': setup_db.javascript_engine(admin),
    'couchdb_url': settings.url,
    'app_user': app.user,
    'dataset': str(DATA_DIR),
    'dataset_orders': (json.loads(manifest_file.read_text())['tables']['service_orders']['rows']
                       if manifest_file.is_file()
                       else sum(1 for _ in open(DATA_DIR / 'service_orders.csv')) - 1),
    'dataset_sha256': ({t: v['sha256'] for t, v in json.loads(manifest_file.read_text())['tables'].items()}
                       if manifest_file.is_file()
                       else {f.stem: sha256(f) for f in sorted(DATA_DIR.glob('*.csv'))}),
    'parameters': {'RUN_LOAD': RUN_LOAD, 'RUN_BENCHMARK': RUN_BENCHMARK, 'RUN_TESTS': RUN_TESTS},
}
(OUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
if environment['git_uncommitted_changes']:
    print('WARNING: uncommitted code changes; results are not tied to one commit:',
          environment['git_uncommitted_changes'])
pd.Series({k: v for k, v in environment.items() if k != 'dataset_sha256'}).to_frame('value')

In [ ]:
def database_overview():
    rows = []
    for name, db in DB.items():
        if not admin.db_exists(db):
            rows.append({'database': db, 'exists': False})
            continue
        info = admin.db_info(db)
        sizes = info.get('sizes') or {}
        rows.append({'database': db, 'exists': True, 'documents': info['doc_count'],
                     'deleted': info['doc_del_count'],
                     'file_MB': round(sizes.get('file', 0) / 2**20, 1),
                     'data_MB': round(sizes.get('external', 0) / 2**20, 1),
                     'part2_tables': ', '.join(DATABASES[name])})
    return pd.DataFrame(rows)

database_overview()

## 2. Load the dataset (optional)

In [ ]:
LOAD_REPORT = OUT_DIR / 'load_report.json'
if RUN_LOAD:
    exit_code = load.main(['--data-dir', str(DATA_DIR), '--reset', '--report', str(LOAD_REPORT)])
    print('load exit code:', exit_code)

if LOAD_REPORT.is_file():
    report = json.loads(LOAD_REPORT.read_text())
    print(f"Loaded from {report['data_dir']} with CouchDB {report['couchdb_version']}, "
          f"batch {report['batch_size']}, {report['workers']} workers")
    display(pd.DataFrame([{'database': db, **v} for db, v in report['databases'].items()]))
    display(pd.DataFrame(report.get('index_builds', [])))
else:
    print('No load report yet: set RUN_LOAD = True to load DATA_DIR.')

## 3. Verify the load (section 3.1)

Every Part 2 table must have the same number of rows in CouchDB (counted by the
`entity_counts` views, e.g. trips embedded in order documents), and a random sample of
documents must turn back into exactly the original CSV rows.

In [ ]:
VERIFY_REPORT = OUT_DIR / 'verify_report.json'
verify_load.main(['--data-dir', str(DATA_DIR), '--sample', '1000', '--report', str(VERIFY_REPORT)])
verification = json.loads(VERIFY_REPORT.read_text())
display(pd.DataFrame(verification['tables']))
pd.DataFrame([{'entity': k, 'sampled': v['sampled'], 'mismatches': len(v['mismatches'])}
              for k, v in verification['samples'].items()])

## 4. Run the essential operations (section 3.2)

Expected results are computed straight from the CSV files of `DATA_DIR`
(`couchdb/reference.py`, which re-implements the Part 2 SQL), and every result is compared
with them. Write operations prepare their starting state, run, are checked by reading the
documents back, and are then undone, so the database ends as it started.

In [ ]:
os.environ['COUCHDB_REFERENCE_FILE'] = str(REFERENCE_FILE)
stale = (not REFERENCE_FILE.is_file()
         or json.loads(REFERENCE_FILE.read_text())['meta']['data_dir'] != str(DATA_DIR))
if stale:
    reference.main(['--data-dir', str(DATA_DIR), '--output', str(REFERENCE_FILE)])
else:
    print(f'Using expected results in {REFERENCE_FILE}')

In [ ]:
from benchmarks.adapters import couchdb as adapter

SCENARIOS = {s.operation_id: s for s in adapter.create_scenarios()}
META = {op.operation_id: op for op in OPERATIONS}
SUMMARY = {}

def as_table(result, rows=PREVIEW_ROWS):
    if isinstance(result, list):
        return pd.DataFrame(result[:rows])
    if isinstance(result, dict) and result.get('type') == 'service_order':
        return Markdown('```json\n' + json.dumps(result, indent=1)[:4000] + '\n```')
    return Markdown('```json\n' + json.dumps(result, indent=1, default=str)[:3000] + '\n```')

def run(operation_id, rows=PREVIEW_ROWS):
    """Run one operation once (setup, execute, validate, teardown) and show its result.
    A failure is recorded in SUMMARY instead of stopping the notebook."""
    scenario, meta = SCENARIOS[operation_id], META[operation_id]
    result, elapsed, passed, error = None, None, False, None
    try:
        ctx = scenario.setup()
        try:
            started = time.perf_counter()
            result = scenario.execute(ctx)
            elapsed = (time.perf_counter() - started) * 1000
            passed = bool(scenario.validate(ctx, result))
        finally:
            scenario.teardown(ctx)
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'[:500]
    count = len(result) if isinstance(result, list) else None
    SUMMARY[operation_id] = {'operation_id': operation_id, 'part2': meta.part2_query,
                             'kind': meta.kind, 'description': meta.description,
                             'requirements': ', '.join(meta.requirement_ids), 'rows': count,
                             'time_ms': round(elapsed, 1) if elapsed is not None else None,
                             'correct': passed, 'error': error}
    head = f'**{operation_id}**' + (f' (Part 2 {meta.part2_query})' if meta.part2_query else '')
    if error:
        display(Markdown(f'{head}: {meta.description}. **ERROR** `{error}`'))
        return None
    display(Markdown(f'{head}: {meta.description}. '
                     + (f'{count:,} rows, ' if count is not None else '')
                     + f'{elapsed:.1f} ms, validation **{"PASSED" if passed else "FAILED"}**'))
    return as_table(result, rows)

### PF-01: Orders and trips (create, read)

In [ ]:
run('PF-01-Q01')

In [ ]:
run('PF-01-Q02')

In [ ]:
run('PF-01-Q03')

### PF-02: Driver assignments

In [ ]:
run('PF-02-Q01')

In [ ]:
run('PF-02-Q02')

In [ ]:
run('PF-02-Q03')

In [ ]:
run('PF-02-Q04')

### PF-03: Service status

In [ ]:
run('PF-03-Q01')

In [ ]:
run('PF-03-Q02')

### PF-04: Payments

In [ ]:
run('PF-04-Q01')

In [ ]:
run('PF-04-Q02')

In [ ]:
run('PF-04-Q03')

### PF-05: Outlet search by area

In [ ]:
run('PF-05-Q01')

### PF-06: Service areas and outlets

In [ ]:
run('PF-06-Q01')

In [ ]:
run('PF-06-Q02')

### PF-07: Deletion

In [ ]:
run('PF-07-Q01')

In [ ]:
run('PF-07-Q02')

### PF-08: Driver and vehicle performance

In [ ]:
run('PF-08-Q01')

In [ ]:
run('PF-08-Q02')

In [ ]:
run('PF-08-Q03')

### PF-09: Activity and demand

In [ ]:
run('PF-09-Q01')

In [ ]:
run('PF-09-Q02')

### PF-10: Payments and revenue

In [ ]:
run('PF-10-Q01')

In [ ]:
run('PF-10-Q02')

In [ ]:
run('PF-10-Q03')

### Summary

In [ ]:
summary = pd.DataFrame(SUMMARY.values()).sort_values('operation_id').reset_index(drop=True)
summary.to_csv(OUT_DIR / 'operations_once.csv', index=False)
print(f"{summary['correct'].sum()}/{len(summary)} operations returned the expected result")
summary

## 5. Query the database yourself

The client wraps the CouchDB HTTP API one request per call, so any request can be tried here.
`DB` holds the database names, e.g. `DB['orders'] == 'grab_orders'`.

In [ ]:
# One order document: subtype, payment, trips and assignments together.
first_order = admin.all_docs(DB['orders'], startkey='ORD', endkey='ORD\ufff0', limit=1)['rows'][0]['id']
admin.get_doc(DB['orders'], first_order)

In [ ]:
# The design documents and their views (section 2.4.1).
pd.DataFrame([{'database': settings.db(name), 'design_doc': ddoc['_id'], 'view': view,
               'reduce': spec.get('reduce', '')}
              for name in DATABASES for ddoc in design.view_ddocs(name)
              for view, spec in ddoc['views'].items()])

In [ ]:
# A view query: trips per service area (grouped _count), like R5 before joining area names.
pd.DataFrame(admin.view(DB['orders'], 'analytics', 'trips_by_start_area', group=True)['rows'])

In [ ]:
# A Mango query and how CouchDB executes it: the index it uses and how much it reads.
query = {
    'selector': {'location.service_area_id': 'SAR0000007', 'outlet_status': 'OPEN'},
    'sort': [{'location.service_area_id': 'asc'}, {'outlet_status': 'asc'},
             {'outlet_name': 'asc'}, {'merchant_outlet_id': 'asc'}],
    'fields': ['merchant_outlet_id', 'outlet_name'],
    'use_index': ['mango-outlets', 'open-outlets-by-area'],
    'limit': 5, 'execution_stats': True,
}
found = admin.find(DB['geo'], query)
explain = admin.json('POST', f"/{DB['geo']}/_explain", body=query)
print('index used:', explain['index']['ddoc'], explain['index']['name'])
print('execution stats:', found.get('execution_stats'))
pd.DataFrame(found['docs'])

In [ ]:
# Write your own query here, e.g.:
# admin.view(DB['orders'], 'ops', 'by_customer', startkey=['CUS0000001'], endkey=['CUS0000001', {}])

## 6. Benchmark (optional)

Runs `benchmarks/run.py` with the CouchDB adapter: one warm-up and three measured runs per
operation, timing only `execute()`. Results go to `benchmarks/results/` (`runs.csv`,
`results.json`); copy them to `benchmarks/published/` once they are final.

In [ ]:
if RUN_BENCHMARK:
    from benchmarks import run as runner
    dataset_size = json.loads((DATA_DIR / 'manifest.json').read_text())['tables']['service_orders']['rows'] \
        if (DATA_DIR / 'manifest.json').is_file() else sum(1 for _ in open(DATA_DIR / 'service_orders.csv')) - 1
    runner.main(['--adapter', 'benchmarks.adapters.couchdb', '--dataset-size', str(dataset_size),
                 '--output-dir', str(OUT_DIR)])

results_file = OUT_DIR / 'results.json'
if results_file.is_file():
    results = json.loads(results_file.read_text())['results']
    table = pd.DataFrame([{
        'operation_id': r['operation_id'], 'part2': META[r['operation_id']].part2_query,
        'scenario': r['scenario'], 'correct': r['validation']['passed'],
        'runs_ms': r['measurements_ms'], 'mean_ms': r['mean_ms'], 'median_ms': r['median_ms'],
    } for r in results if 'operation_id' in r and r.get('dbms') == 'couchdb'])
    table = table.sort_values('operation_id').reset_index(drop=True)
    table.to_csv(OUT_DIR / 'benchmark_summary.csv', index=False)
    print(f"{int(table['correct'].sum())}/{len(table)} scenarios passed in every measured run")
    display(table)
else:
    print('No benchmark results yet: set RUN_BENCHMARK = True.')

## 7. Integrity-rule tests and audit manifest

The tests write valid and invalid documents into temporary databases and check that CouchDB
accepts or rejects each one (Part 2 constraints, status transitions, specialization, ID
immutability, revision conflicts). Their output is saved with the other evidence.

In [ ]:
if RUN_TESTS:
    tests = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_couchdb_rules.py',
                            'tests/test_couchdb_units.py', '-v', '-p', 'no:cacheprovider'],
                           capture_output=True, text=True, cwd=ROOT)
    (OUT_DIR / 'tests.txt').write_text(tests.stdout + tests.stderr)
    print(tests.stdout[-3000:])

In [ ]:
environment['run_finished_utc'] = datetime.now(UTC).isoformat()
(OUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
files = {f.name: {'bytes': f.stat().st_size, 'sha256': sha256(f)}
         for f in sorted(OUT_DIR.iterdir()) if f.is_file() and f.name != 'MANIFEST.json'}
files[REFERENCE_FILE.name] = {'bytes': REFERENCE_FILE.stat().st_size, 'sha256': sha256(REFERENCE_FILE),
                              'note': 'expected results, kept in data/reference (not published)'}
(OUT_DIR / 'MANIFEST.json').write_text(json.dumps(files, indent=2) + '\n')
if PUBLISH:
    target = ROOT / 'benchmarks' / 'published' / OUT_DIR.name
    shutil.copytree(OUT_DIR, target, dirs_exist_ok=True)
    print(f'Evidence copied to {target.relative_to(ROOT)} (commit it to keep it)')
pd.DataFrame(files).T